In [1]:
import re

def get_candidates(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)

    words = text.split()

    # create 1-4 gram phrases (important fix)
    candidates = []
    for i in range(len(words)):
        for j in range(i+1, min(i+5, len(words)+1)):
            candidates.append(" ".join(words[i:j]))

    return candidates

In [2]:
FOOD_CATALOG=[
    "chiken pizza",
    "hot coffee",
    "burger",
    "singara",
    "doi",
    "tea"
]

from sentence_transformers import SentenceTransformer
import numpy as np 

model =SentenceTransformer("all-MiniLM-L6-v2")

catelog_embeddings ={item : model.encode(item ) for item in FOOD_CATALOG}


def cosine(a, b ):
    return np.dot(a, b)/(np.linalg.norm(a)* np.linalg.norm(b))



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
def match_best(candidate):
    vec =model.encode(candidate)

    best_item = None 
    best_score=0


    for item , iv in catelog_embeddings.items():

        score =cosine(vec, iv)
        if score>best_score:
            best_score =score
            best_item= item
    return best_item, best_score

In [4]:
def extract_items(text :str):
    candidates =get_candidates(text)

    found ={}

    for c in candidates:
        item , score =match_best(c)

        if score > 0.7:
            found[item]=item
    return list(found.keys())



In [5]:
extract_items("I want to order a chiken pizza , burger , tea, doi and a hot coffee")

['chiken pizza', 'burger', 'tea', 'doi', 'hot coffee']